##### collapse

In [1]:
from torchvision import datasets, transforms

transform = transforms.ToTensor()

train_data = datasets.CIFAR10(
    root="data",
    train=True,
    download=False,
    transform=transform
)

test_data = datasets.CIFAR10(
    root="data",
    train=False,
    download=False,
    transform=transform
)

In [2]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64)

##### Goal

- introduce Batch Normalization
- make the CNN deeper in a controlled way
- understand why performance improves

### Why CIFAR-10 training feels unstable

- loss goes down, then up
- accuracy improves slowly
- training feels “noisy”

This is not a bug.

Reason
- CIFAR-10 has diverse images
- feature distributions change across layers
- deeper networks amplify this instability

This problem has a name.

### The problem: internal covariate shift

What happens inside a deep network

- layer 1 outputs some distribution of numbers
- layer 2 expects a certain range
- during training, layer 1 keeps changing
- layer 2 constantly has to adapt

So
- learning becomes slower
- gradients become unstable

This is where Batch Normalization comes in.

### What BatchNorm actually does

`nn.BatchNorm2d(C)` does this

- takes activations from a layer
- normalizes them
- keeps mean ≈ 0
- keeps variance ≈ 1
- learns small scale + shift parameters

Important
- this happens per channel
- not per image
- not per pixel

Result
- more stable gradients
- faster convergence
- higher learning rates possible

### Why it is BatchNorm2d, not something else

We use

- `BatchNorm1d` → for linear layers
- `BatchNorm2d` → for convolution outputs
- `BatchNorm3d` → for videos / volumetric data

CNN outputs are

`[batch, channels, height, width]`


So we use

`nn.BatchNorm2d(num_channels)`

### Where BatchNorm goes

correct order:

`Conv2d → BatchNorm2d → ReLU`

Why
- BatchNorm normalizes raw activations
- ReLU then clips negatives


### Make the CNN deeper

We now use two convolution blocks.

Each block
- learns features
- stabilizes them
- then downsamples

### The improved CIFAR-10 CNN

In [3]:
import torch.nn as nn

cnn_model = nn.Sequential(
    nn.Conv2d(3, 32, kernel_size=3, padding=1),
    nn.BatchNorm2d(32),
    nn.ReLU(),
    
    nn.Conv2d(32, 64, kernel_size=3, padding=1),
    nn.BatchNorm2d(64),
    nn.ReLU(),
    
    nn.MaxPool2d(2),
    
    nn.Conv2d(64, 128, kernel_size=3, padding=1),
    nn.BatchNorm2d(128),
    nn.ReLU(),
    
    nn.MaxPool2d(2),
    
    nn.Flatten(),
    nn.Linear(128 * 8 * 8, 10)
)

Input

`[batch, 3, 32, 32]`


After first block
- channels `3 → 32`
- spatial stays `32×32`

`[batch, 32, 32, 32]`


After second block
- channels `32 → 64`
- spatial still `32×32`

`[batch, 64, 32, 32]`


After first pooling

`[batch, 64, 16, 16]`


Third conv block
- channels `64 → 128`
- spatial stays `16×16`

`[batch, 128, 16, 16]`


After second pooling

`[batch, 128, 8, 8]`


Flatten

`128 × 8 × 8 = 8192`


So the Linear layer is:

`nn.Linear(128 * 8 * 8, 10)`


Nothing is arbitrary.

### Loss, Optimizer and GPU

In [4]:
import torch

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(cnn_model.parameters(), lr=0.001)

device = "cuda" if torch.cuda.is_available() else "cpu"
cnn_model = cnn_model.to(device)

### Training

In [5]:
epochs = 10

for epoch in range(epochs):
    cnn_model.train()
    epoch_loss = 0
    
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        preds = cnn_model(images)
        loss = loss_fn(preds, labels)
        
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        epoch_loss += loss.item()
        
    epoch_loss /= len(train_loader)
    print(f"Epoch: {epoch} | Loss: {epoch_loss:.4f}")

Epoch: 0 | Loss: 1.3406
Epoch: 1 | Loss: 0.8842
Epoch: 2 | Loss: 0.7347
Epoch: 3 | Loss: 0.6386
Epoch: 4 | Loss: 0.5556
Epoch: 5 | Loss: 0.4767
Epoch: 6 | Loss: 0.4096
Epoch: 7 | Loss: 0.3402
Epoch: 8 | Loss: 0.2824
Epoch: 9 | Loss: 0.2404


### Testing

In [6]:
cnn_model.eval()
total = 0
correct = 0

for images, labels in test_loader:
    images = images.to(device)
    labels = labels.to(device)
    
    preds = cnn_model(images)
    predicted = preds.argmax(dim=1)
    
    correct += (predicted == labels).sum().item()
    total += labels.size(0)
    
accuracy = correct / total
print(accuracy)

0.7732
